In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import joblib

In [2]:
df = pd.read_csv('df_6month_45difference.csv')
df.head()

,дата выдачи,семейное положение,сумма выдачи,остаток суммы,срок кредита (месяц),цикл,цель кредита,процентная ставка,возраст,пол,...,образование,должность,is_bad_client,year,usd_rate,shaped_bread(som),cottonseed oil(som),beef(som),сумма_выдачи(usd),остаток_суммы(usd)
0,2023-08-11,married,5000000.0,3746309.40,12.0,1.0,миграция,72.0,47.0,м,...,среднее,уста,1,2023,11269,2650,19750,77000,443,332
1,2023-09-05,married,5000000.0,4580890.30,12.0,1.0,миграция,72.0,28.0,м,...,среднее,уста,1,2023,11269,2650,19750,77000,443,406
2,2023-08-18,single,4000000.0,478947.03,12.0,1.0,миграция,72.0,30.0,ж,...,среднее,богбон,1,2023,11269,2650,19750,77000,354,42
3,2023-09-04,single,5000000.0,319060.65,12.0,1.0,миграция,72.0,40.0,м,...,среднее,шифокор,1,2023,11269,2650,19750,77000,443,28
4,2023-09-13,married,4000000.0,2673022.36,12.0,1.0,миграция,72.0,22.0,м,...,среднее,уста,1,2023,11269,2650,19750,77000,354,237


In [3]:
df.columns

Index(['дата выдачи', 'семейное положение', 'сумма выдачи', 'остаток суммы',
       'срок кредита (месяц)', 'цикл', 'цель кредита', 'процентная ставка',
       'возраст', 'пол', 'область \город', 'регион \город',
       'количество членов семьи', 'образование', 'должность', 'is_bad_client',
       'year', 'usd_rate', 'shaped_bread(som)', 'cottonseed oil(som)',
       'beef(som)', 'сумма_выдачи(usd)', 'остаток_суммы(usd)'],
      dtype='object')

In [4]:
df['is_bad_client'].value_counts()

0    3215
1     458
Name: is_bad_client, dtype: int64

In [5]:
bad_client_loans_test = df[df['is_bad_client'] == 1]['остаток_суммы(usd)'].astype(int).sum()
print("Фактическая общая сумма риска в тестовой выборке (USD):", bad_client_loans_test)

Фактическая общая сумма риска в тестовой выборке (USD): 273252


In [6]:
feature_cols = [
    'семейное положение', 'сумма_выдачи(usd)', 'процентная ставка',
    'срок кредита (месяц)', 'цикл', 'цель кредита', 'возраст', 'пол',
    'область \город', 'регион \город', 'количество членов семьи', 'образование',
    'должность', 'shaped_bread(som)', 'cottonseed oil(som)', 'beef(som)'
]

In [7]:
categorical_cols = [
    'семейное положение', 'цель кредита', 'пол', 'область \город',
    'регион \город', 'образование', 'должность'
]

In [8]:
target_col = 'is_bad_client'

In [9]:
obj_df = df[feature_cols].select_dtypes(include=['object']).copy()

In [10]:
def find_unique(col, file):
    # print(col,":",obj_df[col].unique())
    unique_values = obj_df[col].unique()
    file.write(f"{col}: {unique_values}\n")

In [ ]:
# for col in obj_df.columns:
#   find_unique(col)

#save results to a file
with open(output_filename, 'w', encoding='utf-8') as f:
    f.write("Unique values for object columns:\n\n")
    
    for col in obj_df.columns:
        find_unique(col, f)

In [11]:
df_encoded = df.copy()
# label_encoders = {}

In [ ]:
# for col in categorical_cols:
#     le = LabelEncoder()
#     df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
#     label_encoders[col] = le

In [12]:
label_encoders = joblib.load('../model/label_encoders.pkl')

In [13]:
for col in categorical_cols:
    if col in label_encoders:
        le = label_encoders[col]
        #replace unseen labels with a default value (e.g., 'unknown')
        df_encoded[col] = df_encoded[col].astype(str).apply(
            lambda x: x if x in le.classes_ else 'unknown'
        )
        #'unknown' to classes_ if not already present
        if 'unknown' not in le.classes_:
            le.classes_ = np.append(le.classes_, 'unknown')
        df_encoded[col] = le.transform(df_encoded[col])
    else:
        raise ValueError(f"Нет сохранённого LabelEncoder для колонки {col}")

In [14]:
X = df_encoded[feature_cols]
y = df_encoded[target_col]

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

In [16]:
rf_model = joblib.load('../model/6month_random_forest_model.pkl')

In [17]:
y_pred = rf_model.predict(X_test)

In [18]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Confusion Matrix:
[[952  13]
 [ 36 101]]

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.99      0.97       965
           1       0.89      0.74      0.80       137

    accuracy                           0.96      1102
   macro avg       0.92      0.86      0.89      1102
weighted avg       0.95      0.96      0.95      1102



In [19]:
y_proba = rf_model.predict_proba(X_test)[:, 1]

In [20]:
y_test.value_counts()

0    965
1    137
Name: is_bad_client, dtype: int64

In [21]:
npl_ratio = y_test.mean() * 100
print(f"NPL: {npl_ratio:.2f}%")

NPL: 12.43%


In [22]:
print("Classification Report (по 0/1):")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Classification Report (по 0/1):
              precision    recall  f1-score   support

           0       0.96      0.99      0.97       965
           1       0.89      0.74      0.80       137

    accuracy                           0.96      1102
   macro avg       0.92      0.86      0.89      1102
weighted avg       0.95      0.96      0.95      1102

Confusion Matrix:
[[952  13]
 [ 36 101]]


In [23]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, confusion_matrix


thresholds = np.linspace(0, 1, 101)
f1_scores, precisions, recalls, npls, accuracies = [], [], [], [], []
tps, tns, fps, fns = [], [], [], []

thrs = []

for thr in thresholds:
    y_pred_thr = (y_proba >= thr).astype(int)
    
    
    tp, fp, fn, tn = confusion_matrix(y_test, y_pred_thr).ravel()

    tps.append(tp)
    tns.append(tn)
    fps.append(fp)
    fns.append(fn)

    npl = fn / (fn + tp) if (tp + fn) > 0 else 0
    npls.append(round(npl*100, 2))

    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    precisions.append(round(prec *100,2))

    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    recalls.append(round(rec *100, 2))

    f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0
    f1_scores.append(round(f1 *100 ,2))

    acc = (tp + tn) / (tp + tn + fp + fn)
    accuracies.append(round(acc *100 ,2))

    thrs.append(round(thr * 100, 2))

results = pd.DataFrame({
    "threshold": thrs,
    "precision": precisions,
    "recall": recalls,
    "f1": f1_scores,
    "accuracy": accuracies,
    "npl": npls,
    "tp": tps,
    "tn": tns,
    "fp": fps,
    "fn": fns
})

results.to_csv("ehtirom_metrics_.csv", index=False)

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print(f"Лучший порог: {best_threshold:.2f}")
print(f"F1-score: {f1_scores[best_idx]:.4f}")
print(f"Precision: {precisions[best_idx]:.4f}")
print(f"Recall: {recalls[best_idx]:.4f}")
print(f"NPL: {npls[best_idx]:.4f}")
print("ROC-AUC:", roc_auc_score(y_test, y_proba))


Лучший порог: 0.60
F1-score: 97.6500
Precision: 99.1700
Recall: 96.1800
NPL: 3.8200
ROC-AUC: 0.9458832873189364


Then

In [19]:
import pandas as pd
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

# Пороги от 0.1 до 0.99
thresholds = np.round(np.arange(0.1, 1.0, 0.01), 2)
results = []

for threshold in thresholds:
    y_pred_threshold = (y_probs >= threshold).astype(int)

    precision = precision_score(y_test, y_pred_threshold, pos_label=1, zero_division=0)
    recall = recall_score(y_test, y_pred_threshold, pos_label=1, zero_division=0)
    f1 = f1_score(y_test, y_pred_threshold, pos_label=1, zero_division=0)
    accuracy = accuracy_score(y_test, y_pred_threshold)

    results.append({
        'Threshold': round(threshold, 2),
        'Precision': round(precision, 2),
        'Recall': round(recall, 2),
        'F1-Score': round(f1, 2),
        'Accuracy': round(accuracy, 2)
    })

results_df = pd.DataFrame(results)

results_df.to_csv('6month45diff_metrics.csv', index=False, encoding='utf-8')


In [18]:
df['сумма_выдачи(markup)'] = ((df['сумма_выдачи(usd)'] + df['сумма_выдачи(usd)'] * 0.7)).astype(int)

In [19]:
thresholds

array([0.1 , 0.11, 0.12, 0.13, 0.14, 0.15, 0.16, 0.17, 0.18, 0.19, 0.2 ,
       0.21, 0.22, 0.23, 0.24, 0.25, 0.26, 0.27, 0.28, 0.29, 0.3 , 0.31,
       0.32, 0.33, 0.34, 0.35, 0.36, 0.37, 0.38, 0.39, 0.4 , 0.41, 0.42,
       0.43, 0.44, 0.45, 0.46, 0.47, 0.48, 0.49, 0.5 , 0.51, 0.52, 0.53,
       0.54, 0.55, 0.56, 0.57, 0.58, 0.59, 0.6 , 0.61, 0.62, 0.63, 0.64,
       0.65, 0.66, 0.67, 0.68, 0.69, 0.7 , 0.71, 0.72, 0.73, 0.74, 0.75,
       0.76, 0.77, 0.78, 0.79, 0.8 , 0.81, 0.82, 0.83, 0.84, 0.85, 0.86,
       0.87, 0.88, 0.89, 0.9 , 0.91, 0.92, 0.93, 0.94, 0.95, 0.96, 0.97,
       0.98, 0.99])

In [20]:
# Initialize lists for FP, FN, and NPL metrics
lost_profit_markup_fp = []
lost_risk_ostatok_fn = []
predicted_npl_rate = []
predicted_npl_amount = []

# Данные только для теста
df_test = df.loc[X_test.index]  # сохраняем только тестовые записи
total_clients = len(y_test)
total_loan_amount = df_test['остаток_суммы(usd)'].astype(int).sum()

# Calculate actual NPL rate and amount
actual_bad_clients = sum(y_test == 1)
actual_npl_rate = (actual_bad_clients / total_clients) * 100
actual_npl_amount = df_test.loc[y_test == 1, 'остаток_суммы(usd)'].astype(int).sum()

for threshold in thresholds:
    y_pred = (y_probs >= threshold).astype(int)

    # FP (упущенная выгода)
    fp_mask = (y_pred == 1) & (y_test == 0)
    fp_indices = X_test.loc[fp_mask].index
    fp_markup_sum = df_test.loc[fp_indices, 'сумма_выдачи(markup)'].astype(int).sum()
    lost_profit_markup_fp.append(fp_markup_sum)

    # FN (ошибка риска)
    fn_mask = (y_pred == 0) & (y_test == 1)
    fn_indices = X_test.loc[fn_mask].index
    fn_ostatok_sum = df_test.loc[fn_indices, 'остаток_суммы(usd)'].astype(int).sum()
    lost_risk_ostatok_fn.append(fn_ostatok_sum)

    # Predicted NPL rate
    predicted_bad_clients = sum(y_pred == 1)
    npl_rate = (predicted_bad_clients / total_clients) * 100 if total_clients > 0 else 0
    predicted_npl_rate.append(npl_rate)

    # Predicted NPL amount
    predicted_bad_indices = X_test.loc[y_pred == 1].index
    npl_amount = df_test.loc[predicted_bad_indices, 'остаток_суммы(usd)'].astype(int).sum()
    predicted_npl_amount.append(npl_amount)

# Total loss = FP + FN
total_loss = [fp + fn for fp, fn in zip(lost_profit_markup_fp, lost_risk_ostatok_fn)]

# Summary DataFrame
summary_df = pd.DataFrame({
    "Порог": thresholds,
    "FP (Упущ. выгода: markup)": lost_profit_markup_fp,
    "FN (остаток суммы USD)": lost_risk_ostatok_fn,
    "Общая потеря": total_loss,
    "Предсказанный NPL (%)": np.round(predicted_npl_rate, 2),
})

summary_df.to_csv("3month_60diff_losses.csv", index=False)

# Итоги
print("Actual NPL Rate: {:.2f}% ({}/{} clients)".format(actual_npl_rate, actual_bad_clients, total_clients))
print("Actual NPL Amount: {} USD".format(actual_npl_amount))
print("\nModel Performance by Threshold:")
print(summary_df)


Actual NPL Rate: 6.99% (118/1688 clients)
Actual NPL Amount: 74616 USD

Model Performance by Threshold:
    Порог  FP (Упущ. выгода: markup)  FN (остаток суммы USD)  Общая потеря  \
0    0.10                     651713                    2792        654505   
1    0.11                     618219                    2792        621011   
2    0.12                     577160                    3731        580891   
3    0.13                     529811                    3731        533542   
4    0.14                     499507                    3731        503238   
..    ...                        ...                     ...           ...   
85   0.95                          0                   73543         73543   
86   0.96                          0                   73543         73543   
87   0.97                          0                   74616         74616   
88   0.98                          0                   74616         74616   
89   0.99                          0  

Упущенная выгода в тестовой выборке

In [ ]:
test_indices = X_test.index
df_test = df.loc[test_indices]

bad_client_loans_test = df_test[df_test['is_bad_client'] == 1]['остаток_суммы(usd)'].astype(int).sum()
print("Фактическая общая сумма риска в тестовой выборке (USD):", bad_client_loans_test)


Фактическая общая сумма риска в тестовой выборке (USD): 74616


Дерево чтобы показать результаты визуально

In [ ]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score


thresholds = np.arange(0.1, 0.91, 0.1)
auc_score = roc_auc_score(y_test, y_probs)

for threshold in thresholds:
    y_pred = (y_probs >= threshold).astype(int)

    TN = np.sum((y_pred == 0) & (y_test == 0))
    FN = np.sum((y_pred == 0) & (y_test == 1))
    TP = np.sum((y_pred == 1) & (y_test == 1))
    FP = np.sum((y_pred == 1) & (y_test == 0))

    approved = TN + FN
    not_approved = TP + FP

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    npl = round((FN / approved) * 100, 2) if approved > 0 else 0

  
    G = nx.DiGraph()

    #Узлы
    G.add_node("Прогноз модели", pos=(2, 5))

    #Ветви прогноза
    G.add_node("Плохо\n(Positive)", pos=(0, 4))
    G.add_node("Хорошо\n(Negative)", pos=(4, 4))

    # Ветви факта
    G.add_node(f"TP ✅\nПлохой верно\n{TP}", pos=(-1, 3))
    G.add_node(f"FP ❌\nХороший ошибочно\n{FP}", pos=(1, 3))
    G.add_node(f"FN ❌\nПлохой пропущен\n{FN}", pos=(3, 3))
    G.add_node(f"TN ✅\nХороший верно\n{TN}", pos=(5, 3))

    # Связи
    edges = [
        ("Прогноз модели", "Плохо\n(Positive)"),
        ("Прогноз модели", "Хорошо\n(Negative)"),

        ("Плохо\n(Positive)", f"TP ✅\nПлохой верно\n{TP}"),
        ("Плохо\n(Positive)", f"FP ❌\nХороший ошибочно\n{FP}"),

        ("Хорошо\n(Negative)", f"FN ❌\nПлохой пропущен\n{FN}"),
        ("Хорошо\n(Negative)", f"TN ✅\nХороший верно\n{TN}"),
    ]
    G.add_edges_from(edges)

    # Позиции
    pos = nx.get_node_attributes(G, 'pos')

    #Цвета
    node_colors = []
    for node in G.nodes():
        if "✅" in node:
            node_colors.append("#a8e6a3")  # Зелёный
        elif "❌" in node:
            node_colors.append("#f4a6a6")  # Красный
        else:
            node_colors.append("#fff2a8")  # Жёлтый для заголовков

    plt.figure(figsize=(10, 6))
    nx.draw(
        G, pos,
        with_labels=True,
        node_size=4000,
        node_color=node_colors,
        font_size=10,
        edge_color='gray'
    )

    metrics_text = f"AUC: {auc_score:.2f}\nF1: {f1:.2f}\nAcc: {acc:.2f}\nNPL: {npl} %"
    plt.gcf().text(0.75, 0.8, metrics_text, fontsize=12,
                   bbox=dict(facecolor='lightgrey', alpha=0.3))

    plt.title(f"Дерево решений при threshold = {threshold:.2f}", fontsize=14)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
